# Load Model

In [ ]:
import re
import os
import random
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login
from peft import PeftModel
import time
import json
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
token = user_secrets.get_secret("HF_TOKEN")

seed = 42
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

login(token=token)

MODEL_ID = "CohereLabs/tiny-aya-base"
ADAPTER_REPO = "legesher/language-decoded-lora-condition-2-ur-5k"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = "left"  # required for correct batch generation in decoder-only models
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def load_model(merge_adapter=True):
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto"
    )
    print(f"Loading adapter")
    model = PeftModel.from_pretrained(base_model, ADAPTER_REPO)
    if merge_adapter:
        print("Merging adapter into base model for inference...")
        model = model.merge_and_unload()
    else:
        print("Keeping model as PEFT model (base + adapter not merged).")
    model.eval()
    return model

Load adapters

In [ ]:
model = load_model(merge_adapter=True)

In [ ]:
import time

start = time.time()
from tqdm import tqdm

# ─────────────────────────────────────────────
# BATCHED GENERATION  (replaces generate_text)
# ─────────────────────────────────────────────
def generate_texts_batch(prompts: list[str], max_new_tokens: int = 80, batch_size: int = 32, desc: str = "") -> list[str]:
    all_outputs = []
    model_device = next(model.parameters()).device
    batches = range(0, len(prompts), batch_size)

    for i in tqdm(batches, desc=desc or "Generating", unit="batch"):
        batch = prompts[i : i + batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=256)
        inputs = {k: v.to(model_device) for k, v in inputs.items()}
        input_len = inputs["input_ids"].shape[1]

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )

        for output in outputs:
            text = tokenizer.decode(output[input_len:], skip_special_tokens=True).strip()
            all_outputs.append(text)

    return all_outputs

**Load datasets**

SIB-200

In [ ]:
SIB200_DATASET = "Davlan/sib200"
SIB200_CONFIGS = {
    "es": "spa_Latn",
    "zh": "zho_Hans",
    "ur": "urd_Arab",
}
SIB200_FALLBACK_LABELS = [
    "science/technology",
    "travel",
    "politics",
    "sports",
    "health",
    "entertainment",
    "geography",
]

sib200_datasets = {
    lang: load_dataset(SIB200_DATASET, config)
    for lang, config in SIB200_CONFIGS.items()
}

sib200_features = sib200_datasets["es"]["test"].features
if "label" in sib200_features and hasattr(sib200_features["label"], "names"):
    SIB200_LABELS = sib200_features["label"].names
    SIB200_GOLD_COLUMN = "label"
elif "category" in sib200_features:
    SIB200_LABELS = SIB200_FALLBACK_LABELS
    SIB200_GOLD_COLUMN = "category"
else:
    raise ValueError(f"Unsupported SIB-200 schema: {sib200_features}")

sib200_zh = sib200_datasets["zh"]["test"].to_pandas()
sib200_es = sib200_datasets["es"]["test"].to_pandas()
sib200_ur = sib200_datasets["ur"]["test"].to_pandas()


Belebele

In [ ]:
BELEBELE_DATASET = "facebook/belebele"
BELEBELE_CONFIGS = {
    "es": "spa_Latn",
    "zh": "zho_Hans",
    "ur": "urd_Arab",
}

belebele_zh = load_dataset(BELEBELE_DATASET, BELEBELE_CONFIGS["zh"])["test"].to_pandas()
belebele_es = load_dataset(BELEBELE_DATASET, BELEBELE_CONFIGS["es"])["test"].to_pandas()
belebele_ur = load_dataset(BELEBELE_DATASET, BELEBELE_CONFIGS["ur"])["test"].to_pandas()


CSQA

In [ ]:
csqa_es = load_dataset("INK-USC/xcsr", "X-CSQA-es")["validation"].to_pandas()
csqa_zh = load_dataset("INK-USC/xcsr", "X-CSQA-zh")["validation"].to_pandas()
csqa_ur = load_dataset("INK-USC/xcsr", "X-CSQA-ur")["validation"].to_pandas()

def prepare_csqa(df):
    df = df[["question", "answerKey"]].copy()
    df["stem"] = df["question"].apply(lambda x: x["stem"])
    df["choice_texts"] = df["question"].apply(lambda x: list(x["choices"]["text"]))
    df["A"] = df["choice_texts"].apply(lambda x: x[0])
    df["B"] = df["choice_texts"].apply(lambda x: x[1])
    df["C"] = df["choice_texts"].apply(lambda x: x[2])
    df["D"] = df["choice_texts"].apply(lambda x: x[3])
    df["E"] = df["choice_texts"].apply(lambda x: x[4])
    return df[["stem", "A", "B", "C", "D", "E", "answerKey"]]

csqa_es = prepare_csqa(csqa_es)
csqa_zh = prepare_csqa(csqa_zh)
csqa_ur = prepare_csqa(csqa_ur)

SIB-200 Functions

In [ ]:
SIB200_OPTION_LABELS = list("ABCDEFG")
SIB200_TOPIC_TO_LETTER = {
    topic: SIB200_OPTION_LABELS[index]
    for index, topic in enumerate(SIB200_LABELS)
}
SIB200_LETTER_TO_TOPIC = {
    letter: topic
    for topic, letter in SIB200_TOPIC_TO_LETTER.items()
}


def format_sib200_options() -> str:
    return "\n".join(
        f"{SIB200_TOPIC_TO_LETTER[topic]}. {topic}"
        for topic in SIB200_LABELS
    )


def build_sib200_prompt(text: str, lang: str, template_id: int) -> str:
    options = format_sib200_options()
    templates = {
        1: {
            "es": f"""Clasifica la oración de noticias en un tema.
Responde solo con una letra.

Oración: {text}

{options}

Respuesta:""",
            "zh": f"""请将下面的新闻句子分类到一个主题。
只回答一个字母。

句子: {text}

{options}

答案:""",
            "ur": f"""نیچے دیے گئے خبروں کے جملے کو ایک موضوع میں درجہ بند کریں۔
صرف ایک حرف میں جواب دیں۔

جملہ: {text}

{options}

جواب:""",
        },
        2: {
            "es": f"""Lee la siguiente oración de noticias y elige su categoría principal.
Contesta únicamente con la letra de la opción correcta.

Texto: {text}

{options}

Opción:""",
            "zh": f"""阅读下面的新闻句子，选择它最合适的主题类别。
请只输出正确选项的字母。

文本: {text}

{options}

选项:""",
            "ur": f"""درج ذیل خبر کے جملے کو پڑھ کر اس کا بنیادی موضوع منتخب کریں۔
درست اختیار کا صرف حرف لکھیں۔

متن: {text}

{options}

اختیار:""",
        },
    }
    return templates[template_id][lang]


def extract_sib200_choice(text: str):
    text = text.strip().upper()
    valid = "".join(SIB200_OPTION_LABELS[:len(SIB200_LABELS)])
    match = re.search(rf"\b([{valid}])\b", text)
    if match:
        return match.group(1)
    match = re.search(rf"ANSWER\s*[:\-]?\s*([{valid}])", text)
    if match:
        return match.group(1)
    return None


def normalize_sib200_gold(label):
    if isinstance(label, str) and not label.isdigit():
        return SIB200_TOPIC_TO_LETTER.get(label.strip())
    return SIB200_OPTION_LABELS[int(label)]


def evaluate_sib200(df: pd.DataFrame, lang: str, template_id: int, n_samples=None, batch_size: int = 32):
    eval_df = df if n_samples is None else df.head(n_samples)

    prompts = [
        build_sib200_prompt(row["text"], lang, template_id)
        for _, row in eval_df.iterrows()
    ]
    outputs = generate_texts_batch(
        prompts,
        max_new_tokens=10,
        batch_size=batch_size,
        desc=f"SIB-200 {lang} template {template_id}",
    )

    rows = []
    correct = 0
    for (_, row), output in zip(eval_df.iterrows(), outputs):
        pred = extract_sib200_choice(output)
        gold = normalize_sib200_gold(row[SIB200_GOLD_COLUMN])
        is_correct = pred == gold
        correct += int(is_correct)
        rows.append({
            "template_id": template_id,
            "language": lang,
            "text": row["text"],
            "raw_output": output,
            "pred": pred,
            "pred_topic": SIB200_LETTER_TO_TOPIC.get(pred),
            "gold": gold,
            "gold_topic": SIB200_LETTER_TO_TOPIC.get(gold),
            "correct": is_correct,
        })

    return correct / len(rows) if rows else 0.0, pd.DataFrame(rows)


Belebele Functions

In [ ]:
def build_belebele_prompt(row, lang: str, template_id: int) -> str:
    templates = {
        1: {
            "es": f"""Lee el pasaje y responde la pregunta.
Responde solo con una letra: A, B, C o D.

Pasaje: {row['flores_passage']}

Pregunta: {row['question']}

A. {row['mc_answer1']}
B. {row['mc_answer2']}
C. {row['mc_answer3']}
D. {row['mc_answer4']}

Respuesta:""",
            "zh": f"""请阅读文章并回答问题。
只回答一个字母：A、B、C 或 D。

文章: {row['flores_passage']}

问题: {row['question']}

A. {row['mc_answer1']}
B. {row['mc_answer2']}
C. {row['mc_answer3']}
D. {row['mc_answer4']}

答案:""",
            "ur": f"""اقتباس پڑھیں اور سوال کا جواب دیں۔
صرف ایک حرف میں جواب دیں: A، B، C، یا D۔

اقتباس: {row['flores_passage']}

سوال: {row['question']}

A. {row['mc_answer1']}
B. {row['mc_answer2']}
C. {row['mc_answer3']}
D. {row['mc_answer4']}

جواب:""",
        },
        2: {
            "es": f"""Usa el pasaje para elegir la mejor respuesta a la pregunta.
Escribe solamente una letra: A, B, C o D.

Texto: {row['flores_passage']}

Pregunta: {row['question']}

A. {row['mc_answer1']}
B. {row['mc_answer2']}
C. {row['mc_answer3']}
D. {row['mc_answer4']}

Opción:""",
            "zh": f"""根据下面的文章，选择问题的最佳答案。
请只写一个字母：A、B、C 或 D。

文本: {row['flores_passage']}

问题: {row['question']}

A. {row['mc_answer1']}
B. {row['mc_answer2']}
C. {row['mc_answer3']}
D. {row['mc_answer4']}

选项:""",
            "ur": f"""دیے گئے اقتباس کی بنیاد پر سوال کا بہترین جواب منتخب کریں۔
صرف ایک حرف لکھیں: A، B، C، یا D۔

متن: {row['flores_passage']}

سوال: {row['question']}

A. {row['mc_answer1']}
B. {row['mc_answer2']}
C. {row['mc_answer3']}
D. {row['mc_answer4']}

اختیار:""",
        },
    }
    return templates[template_id][lang]


def extract_belebele_choice(text: str):
    text = text.strip().upper()
    match = re.search(r"\b([ABCD])\b", text)
    if match:
        return match.group(1)
    match = re.search(r"ANSWER\s*[:\-]?\s*([ABCD])", text)
    if match:
        return match.group(1)
    return None


def normalize_belebele_gold(answer_num):
    return "ABCD"[int(answer_num) - 1]


def evaluate_belebele(df: pd.DataFrame, lang: str, template_id: int, n_samples=None, batch_size: int = 32):
    eval_df = df if n_samples is None else df.head(n_samples)

    prompts = [
        build_belebele_prompt(row, lang, template_id)
        for _, row in eval_df.iterrows()
    ]
    outputs = generate_texts_batch(
        prompts,
        max_new_tokens=10,
        batch_size=batch_size,
        desc=f"Belebele {lang} template {template_id}",
    )

    rows = []
    correct = 0
    for (_, row), output in zip(eval_df.iterrows(), outputs):
        pred = extract_belebele_choice(output)
        gold = normalize_belebele_gold(row["correct_answer_num"])
        is_correct = pred == gold
        correct += int(is_correct)
        rows.append({
            "template_id": template_id,
            "language": lang,
            "passage": row["flores_passage"],
            "question": row["question"],
            "raw_output": output,
            "pred": pred,
            "gold": gold,
            "correct": is_correct,
        })

    return correct / len(rows) if rows else 0.0, pd.DataFrame(rows)


CSQA Functions

In [ ]:
def build_csqa_prompt(row, lang: str, template_id: int) -> str:
    templates = {
        1: {
            "es": f"""Elige la mejor respuesta para la siguiente pregunta de sentido común.
Responde solo con una letra: A, B, C, D o E.
Pregunta: {row['stem']}
A. {row['A']}
B. {row['B']}
C. {row['C']}
D. {row['D']}
E. {row['E']}
Respuesta:""",
            "zh": f"""请选择下面这个常识问题的最佳答案。
只回答一个字母：A、B、C、D 或 E。
问题: {row['stem']}
A. {row['A']}
B. {row['B']}
C. {row['C']}
D. {row['D']}
E. {row['E']}
答案:""",
            "ur": f"""نیچے دیے گئے عمومی فہم کے سوال کا بہترین جواب منتخب کریں۔
صرف ایک حرف میں جواب دیں: A، B، C، D، یا E۔
سوال: {row['stem']}
A. {row['A']}
B. {row['B']}
C. {row['C']}
D. {row['D']}
E. {row['E']}
جواب:""",
        },
        2: {
            "es": f"""Lee la pregunta de sentido común y selecciona la opción más adecuada.
Contesta únicamente con una letra: A, B, C, D o E.
Pregunta: {row['stem']}
A. {row['A']}
B. {row['B']}
C. {row['C']}
D. {row['D']}
E. {row['E']}
Opción:""",
            "zh": f"""阅读这个常识问题，并选择最合适的答案。
请只输出一个字母：A、B、C、D 或 E。
问题: {row['stem']}
A. {row['A']}
B. {row['B']}
C. {row['C']}
D. {row['D']}
E. {row['E']}
选项:""",
            "ur": f"""عام فہم کے اس سوال کو پڑھیں اور سب سے مناسب جواب چنیں۔
صرف ایک حرف لکھیں: A، B، C، D، یا E۔
سوال: {row['stem']}
A. {row['A']}
B. {row['B']}
C. {row['C']}
D. {row['D']}
E. {row['E']}
اختیار:""",
        },
    }
    return templates[template_id][lang]


def extract_choice(text: str):
    text = text.strip().upper()
    match = re.search(r"\b([ABCDE])\b", text)
    if match:
        return match.group(1)
    match = re.search(r"ANSWER\s*[:\-]?\s*([ABCDE])", text)
    if match:
        return match.group(1)
    return None


def evaluate_csqa(df: pd.DataFrame, lang: str, template_id: int, n_samples=None, batch_size: int = 32):
    eval_df = df if n_samples is None else df.head(n_samples)

    prompts = [
        build_csqa_prompt(row, lang, template_id)
        for _, row in eval_df.iterrows()
    ]
    outputs = generate_texts_batch(
        prompts,
        max_new_tokens=10,
        batch_size=batch_size,
        desc=f"X-CSQA {lang} template {template_id}",
    )

    rows = []
    correct = 0
    for (_, row), output in zip(eval_df.iterrows(), outputs):
        pred = extract_choice(output)
        gold = str(row["answerKey"]).strip().upper()
        is_correct = pred == gold
        correct += int(is_correct)
        rows.append({
            "template_id": template_id,
            "language": lang,
            "stem": row["stem"],
            "raw_output": output,
            "pred": pred,
            "gold": gold,
            "correct": is_correct,
        })

    return correct / len(rows) if rows else 0.0, pd.DataFrame(rows)


Evals

In [ ]:
def results_to_jsonable(results):
    jsonable = {"summary": results["summary"]}
    for key, value in results.items():
        if key == "summary":
            continue
        jsonable[key] = value.to_dict(orient="records") if value is not None else None
    return jsonable

All evals

In [ ]:
from huggingface_hub import HfApi
import json

cond = "condition-2-ur-5k"
TEMPLATE_IDS = [1, 2]
LANGUAGE_BENCHMARKS = {
    "zh": {
        "sib200": sib200_zh,
        "belebele": belebele_zh,
        "csqa": csqa_zh,
    },
    "es": {
        "sib200": sib200_es,
        "belebele": belebele_es,
        "csqa": csqa_es,
    },
    "ur": {
        "sib200": sib200_ur,
        "belebele": belebele_ur,
        "csqa": csqa_ur,
    },
}


def evaluate_template(template_id: int, batch_size: int = 32):
    summary = {}
    results = {"summary": summary, "template_id": template_id}

    for lang, datasets in LANGUAGE_BENCHMARKS.items():
        summary[f"sib200_{lang}_acc"], results[f"sib200_{lang}"] = evaluate_sib200(
            datasets["sib200"], lang=lang, template_id=template_id, batch_size=batch_size
        )
        print(f"SIB-200 Done - {lang} - Template {template_id}")

        summary[f"belebele_{lang}_acc"], results[f"belebele_{lang}"] = evaluate_belebele(
            datasets["belebele"], lang=lang, template_id=template_id, batch_size=batch_size
        )
        print(f"Belebele Done - {lang} - Template {template_id}")

        summary[f"csqa_{lang}_acc"], results[f"csqa_{lang}"] = evaluate_csqa(
            datasets["csqa"], lang=lang, template_id=template_id, batch_size=batch_size
        )
        print(f"X-CSQA Done - {lang} - Template {template_id}")

    return results


def save_results(results, summary_path, full_path):
    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(
            {
                "template_id": results["template_id"],
                "summary": results["summary"],
            },
            f,
            ensure_ascii=False,
            indent=2,
        )

    json_results = {
        k: (v.to_dict(orient="records") if isinstance(v, pd.DataFrame) else v)
        for k, v in results.items()
    }
    with open(full_path, "w", encoding="utf-8") as f:
        json.dump(json_results, f, ensure_ascii=False, indent=2)


if __name__ == "__main__":
    start = time.time()
    BATCH_SIZE = 256

    api = HfApi()
    for template_id in TEMPLATE_IDS:
        template_results = evaluate_template(template_id=template_id, batch_size=BATCH_SIZE)

        summary_path = f"/kaggle/working/{cond}_template_{template_id}_summary.json"
        full_path = f"/kaggle/working/{cond}_template_{template_id}_results.json"
        save_results(template_results, summary_path, full_path)

        for local_path, path_in_repo in [
            (summary_path, f"conditions/{cond}/results/template_{template_id}_summary.json"),
            (full_path, f"conditions/{cond}/results/template_{template_id}_results.json"),
        ]:
            api.upload_file(
                path_or_fileobj=local_path,
                path_in_repo=path_in_repo,
                repo_id="legesher/language-decoded-experiments",
                repo_type="dataset",
            )

    end = time.time()
    print(f"Time taken: {end - start:.2f} seconds")
